# Lab 4 – Context Engineering & Knowledge Structuring

**Ziel (Tag 1: 25 Min + Tag 2: 20 Min):** Verstehen, dass die Qualität eines RAG-Systems *vor* dem Embedding
entschieden wird – beim Parsen, Chunken und Packen des Kontexts.

**Teil 1 (Tag 1):** Parsing (pypdf vs. Docling), struktur-erhaltendes Chunking, Tabellen, Chunk-Größe
**Teil 2 (Tag 2):** Parent-Child-Retrieval, Context Packing, Re-Ordering, Long-Context vs. RAG

In [ ]:
import sys, os, time
_root = os.getcwd()                      # Repo-Root finden (Notebook liegt in labs/ oder labs/solutions/)
while not os.path.isdir(os.path.join(_root, "ragkurs")):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root); os.chdir(_root)
import pandas as pd
pd.set_option("display.max_colwidth", 90)

from ragkurs import load_corpus, chunk_fixed, chunk_by_headings, chunk_parent_child, HybridIndex, RAGPipeline, PipelineConfig
from ragkurs.eval import load_golden, evaluate_retrieval, retrieval_summary, compare_retrieval, run_golden, e2e_summary

golden = load_golden()
tabellen = [g for g in golden if g["type"] == "tabelle"]

## Teil A – Walkthrough (Tag 1)

### A1 Parsing entscheidet: pypdf vs. Docling

Docling (IBM, MIT-Lizenz) erkennt Layout, Überschriften und Tabellen und liefert Markdown. Der erste Aufruf
lädt die Modelle (~500 MB) – auf vorbereiteten VMs sind sie im Cache.

In [ ]:
from ragkurs.loading import load_document
from pathlib import Path

pdf = Path("data/corpus/prod-handbuch-ax200.pdf")
naive = load_document(pdf, pdf_parser="pypdf")
t0 = time.perf_counter(); structured = load_document(pdf, pdf_parser="docling"); print(f"Docling: {time.perf_counter()-t0:.1f}s")

print("=== pypdf ===");   print(naive.text[900:1500])
print("\n=== Docling ==="); print(structured.text[900:1900])

### A2 Der Korpus mit Docling – und was das mit dem Chunking macht

In [ ]:
docs_pypdf = load_corpus(pdf_parser="pypdf")
docs_docling = load_corpus(pdf_parser="docling")     # nur die 5 PDFs laufen durch Docling, Markdown bleibt Markdown

for name, docs in (("pypdf", docs_pypdf), ("docling", docs_docling)):
    ch = chunk_by_headings(docs)
    pdf_chunks = [c for c in ch if c.doc_id == "prod-handbuch-ax200"]
    print(f"{name:8s}: {len(ch)} Chunks gesamt, AX-200-Handbuch: {len(pdf_chunks)} Chunks, Abschnitte: {[c.section.split(' > ')[-1][:25] for c in pdf_chunks]}")

In [ ]:
# Ein Tabellen-Chunk bleibt als Ganzes erhalten - mit Breadcrumb:
wart = next(c for c in chunk_by_headings(docs_docling) if c.doc_id == "prod-handbuch-ax200" and "Wartung" in c.section)
print(wart.text[:900])

### A3 Der Effekt auf Tabellenfragen (nur Retrieval, ohne LLM)

In [ ]:
variants = {
    "pypdf+fixed":      chunk_fixed(docs_pypdf, 800, 100),
    "pypdf+headings":   chunk_by_headings(docs_pypdf),
    "docling+headings": chunk_by_headings(docs_docling),
}
pipes = [RAGPipeline(HybridIndex(collection=n.replace("+", "_")).build(ch), PipelineConfig(retrieval="hybrid", k=5, name=n)) for n, ch in variants.items()]
compare_retrieval(pipes, tabellen, user_roles=["employee"])

## Teil B – Aufgaben (Tag 1)

### B1 Chunk-Größe sweepen
Variiert `max_chars` in `chunk_by_headings` (400, 800, 1500, 3000) auf den Docling-Dokumenten.
Tragt Hit-Rate, Anzahl Chunks und durchschnittliche Kontextlänge (Zeichen der Top-5) auf.
Wo liegt der Sweet Spot – und warum ist „größer“ nicht automatisch besser?

In [ ]:
# HINWEIS: Kontextlänge = sum(len(h.chunk.text) for h in pipe.retrieve_only(q).hits)
# TODO: hier implementieren (Loesung: labs/solutions/)
...

### B2 End-to-End auf Tabellenfragen
Lasst `run_golden` (mit Judge) auf den 17 Tabellenfragen laufen – einmal `pypdf+fixed`, einmal `docling+headings`.
Wie ändert sich die **Antwort-Korrektheit** (nicht nur die Hit-Rate)?

In [ ]:
# HINWEIS: run_golden(pipe, tabellen, ["employee"], verbose=False) -> e2e_summary(df)
# TODO: hier implementieren (Loesung: labs/solutions/)
...

### B3 Breadcrumb-Ablation
`chunk_by_headings(docs, breadcrumb=False)` entfernt die `[Dokument > Abschnitt]`-Zeile. Messt den Effekt auf
`near-miss`-Fragen. Warum hilft der Breadcrumb gerade dort?

In [ ]:
# HINWEIS: near_miss = [g for g in golden if g["type"] == "near-miss"]
# TODO: hier implementieren (Loesung: labs/solutions/)
...

---
## Teil A – Walkthrough (Tag 2)

### A4 Parent-Child: klein suchen, groß liefern

Kleine Chunks (~350 Zeichen) treffen präziser, große Chunks (ganze Abschnitte) geben dem LLM den nötigen Kontext.
Also: Kinder indexieren, beim Antworten den Elternabschnitt liefern.

In [ ]:
children, parents = chunk_parent_child(docs_docling, parent_chars=1500, child_chars=350)
print(len(children), "Kinder,", len(parents.parents), "Eltern")
pc_index = HybridIndex(collection="parent_child").build(children)

pc = RAGPipeline(pc_index, PipelineConfig(retrieval="hybrid", k=5, prefetch_k=20, parent_expand=True, name="parent-child"), parent_store=parents)
r = pc.retrieve_only("Wie hoch ist die Verpflegungspauschale bei 24 Stunden Abwesenheit?", ["employee"])
for h in r.hits[:3]:
    print(h.source, h.chunk.chunk_id, "| Kind:", h.extra.get("child_id"), "|", len(h.chunk.text), "Zeichen")

In [ ]:
flat = RAGPipeline(HybridIndex(collection="flat").build(chunk_by_headings(docs_docling)), PipelineConfig(retrieval="hybrid", k=5, prefetch_k=20, name="headings-flat"))
compare_retrieval([flat, pc], golden, ["employee"])

### A5 Context Packing und Re-Ordering

`build_context` dedupliziert (Kinder desselben Elternteils), hält ein Zeichenbudget ein und kann die Reihenfolge
ändern: **Lost in the Middle** (Liu et al. 2023) – Modelle nutzen Anfang und Ende des Kontexts besser als die Mitte.

In [ ]:
from ragkurs.generate import build_context, lost_in_the_middle_reorder

hits = flat.retrieve_only("Welche Reaktionszeit gilt für einen Incident der Stufe S2?", ["employee"]).hits
print("Original    :", [h.rank for h in hits])
print("Re-Ordering :", [h.rank for h in lost_in_the_middle_reorder(hits)])
ctx, used = build_context(hits, max_chars=2500, reorder="lost_in_middle")
print(len(ctx), "Zeichen,", len(used), "Chunks verwendet")

## Teil B – Aufgaben (Tag 2)

### B4 Re-Ordering messen
Vergleicht end-to-end (Judge) `reorder="none"` vs. `"lost_in_middle"` bei **k = 10** auf 15 Fragen.
Bei k=5 ist der Effekt meist klein – warum?

In [ ]:
# HINWEIS: PipelineConfig(retrieval="hybrid", k=10, prefetch_k=25, reorder=..., max_context_chars=12000)
# TODO: hier implementieren (Loesung: labs/solutions/)
...

### B5 Long-Context statt RAG?
Der gesamte Korpus hat ~55.000 Zeichen (~15.000 Tokens) – das passt locker in ein modernes Kontextfenster.
Baut eine „Alles-in-den-Kontext“-Variante: ein Hit pro Dokument, `max_chars` sehr groß, und vergleicht auf
10 Fragen **Korrektheit, Latenz und Kosten** mit der RAG-Pipeline. Wann kippt die Rechnung (Stichworte:
Korpusgröße, Anfragen/Tag, Context Rot, Zugriffsrechte)?

In [ ]:
# HINWEIS: from ragkurs.index import Hit ; from ragkurs.chunking import Chunk ; generate.answer(q, hits, max_chars=10**7)
# TODO: hier implementieren (Loesung: labs/solutions/)
...

## Teil C – Debrief

1. Welcher Schritt hat auf Tabellenfragen mehr gebracht – Docling oder das Chunking? Was folgt daraus für die
   Reihenfolge, in der ihr ein System optimiert?
2. Parent-Child erhöht die Kontextlänge. Wo ist die Grenze, ab der Context Noise den Gewinn auffrisst?
3. Long-Context: Bei welcher Korpusgröße und Anfragezahl würdet ihr komplett auf Retrieval verzichten?
   (Und was ist mit Zugriffsrechten?)

**Merksatz:** Garbage in, garbage out gilt für RAG doppelt: Was das Parsing zerstört, findet kein Embedding wieder.